# A Dozen Boxes, One Object

**Topic:** IoU and non-maximum suppression &nbsp;·&nbsp; **Difficulty:** Medium

A detector fires on every anchor that overlaps an object, so a single car comes
back as 8 to 14 boxes at slightly different scales. Non-maximum suppression is the
greedy cleanup: keep the highest-scoring box, delete everything that overlaps it
too much, repeat.

Implement IoU and NMS from scratch. The detail people get wrong is that IoU needs
the intersection clamped at zero: boxes that do not touch produce a negative
width if you subtract carelessly.

## The data (created in the next cell)

- `boxes` `(120, 4)`: `[x1, y1, x2, y2]` with `x2 > x1`, `y2 > y1`.
- `scores` `(120,)`: detector confidences.
- Boxes cluster around 8 true objects, plus some isolated false positives.

## Your task

With **NumPy only**:

1. `iou_matrix(boxes)`: the full `(120, 120)` IoU matrix. Clamp intersection
   width and height at zero.
2. `nms(boxes, scores, thresh)`: greedy NMS at IoU threshold `0.5`, returning
   kept indices in the order they were selected (highest score first).
3. Set `answers` to a dict with `"n_kept"` (int), `"kept_scores"` (the scores of
   the kept boxes, in selection order), `"max_iou_among_kept"` (the largest IoU
   between any two kept boxes), and `"n_kept_thresh_09"` (kept count at threshold
   `0.9`).

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks no two kept boxes exceed the threshold and that a looser threshold keeps
more boxes.

In [ ]:
import numpy as np

rng = np.random.default_rng(3015)
centres = rng.uniform(50, 450, (8, 2))
boxes, scores = [], []
for c in centres:
    for _ in range(rng.integers(8, 15)):
        w, h = rng.uniform(40, 70), rng.uniform(40, 70)
        j = rng.normal(0, 6, 2)
        boxes.append([c[0] - w / 2 + j[0], c[1] - h / 2 + j[1],
                      c[0] + w / 2 + j[0], c[1] + h / 2 + j[1]])
        scores.append(rng.uniform(0.3, 0.99))
while len(boxes) < 120:
    c = rng.uniform(0, 500, 2)
    boxes.append([c[0], c[1], c[0] + 30, c[1] + 30])
    scores.append(rng.uniform(0.05, 0.3))
boxes = np.array(boxes[:120], dtype=float)
scores = np.array(scores[:120], dtype=float)
print(f"boxes {boxes.shape}, clustered around {len(centres)} true objects")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("n_kept", [], 65.0, -10.883470962942445, 1e-06),
    ("kept_scores", [65], 21.07356161132662, 0.5436933129437471, 1e-06),
    ("max_iou_among_kept", [], 0.4996564051370139, -0.08366147656549074, 1e-06),
    ("n_kept_thresh_09", [], 118.0, -19.7576857481109, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "iou_matrix" not in dir() or "nms" not in dir():
    raise NameError("Define `iou_matrix(boxes)` and `nms(boxes, scores, thresh)` in the solution cell.")
_IOU = np.asarray(iou_matrix(boxes), dtype=float)
if _IOU.shape != (120, 120) or not np.all(np.isclose(np.diag(_IOU), 1.0, atol=1e-9)):
    print("  iou_matrix              wrong")
    _ok = False
_keep = list(nms(boxes, scores, 0.5))
if len(_keep) != int(answers.get("n_kept", -1)):
    print("  nms                     wrong")
    _ok = False
elif any(scores[_keep[j]] < scores[_keep[j + 1]] for j in range(len(_keep) - 1)):
    print("  nms                     wrong (kept indices not in non-increasing score order)")
    _ok = False

if "max_iou_among_kept" in answers:
    _sup = answers["max_iou_among_kept"] <= 0.5 + 1e-9
    _more = int(answers["n_kept_thresh_09"]) > int(answers["n_kept"])
    print(f"  {'no kept pair over 0.5':<24} {'ok' if _sup else 'wrong'}  "
          f"(max {answers['max_iou_among_kept']:.4f})")
    print(f"  {'looser keeps more':<24} {'ok' if _more else 'wrong'}  "
          f"({int(answers['n_kept'])} at 0.5, {int(answers['n_kept_thresh_09'])} at 0.9)")
    _ok = _ok and _sup and _more

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

The clamp is the whole trick: `max(0, min(x2a, x2b) - max(x1a, x1b))` is the
overlap width, and without the `max(0, ...)` two disjoint boxes produce a
negative width that multiplies with a negative height into a *positive* fake
intersection. Greedy NMS is `O(n^2)` in the worst case and is what almost every
detector ships. The threshold trades duplicate suppression against missing
genuinely overlapping objects, which is why crowded scenes use Soft-NMS
instead.

```python
import numpy as np

def iou_matrix(b):
    x1 = np.maximum(b[:, None, 0], b[None, :, 0])
    y1 = np.maximum(b[:, None, 1], b[None, :, 1])
    x2 = np.minimum(b[:, None, 2], b[None, :, 2])
    y2 = np.minimum(b[:, None, 3], b[None, :, 3])
    iw = np.clip(x2 - x1, 0, None)
    ih = np.clip(y2 - y1, 0, None)
    inter = iw * ih
    area = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    union = area[:, None] + area[None, :] - inter
    return inter / np.maximum(union, 1e-12)

IOU = iou_matrix(boxes)

def nms(boxes, scores, thresh):
    iou = iou_matrix(boxes)
    order = np.argsort(-scores)
    keep = []
    alive = np.ones(len(scores), dtype=bool)
    for i in order:
        if not alive[i]:
            continue
        keep.append(int(i))
        alive &= iou[i] <= thresh
        alive[i] = False
    return keep

kept = nms(boxes, scores, 0.5)
sub = IOU[np.ix_(kept, kept)].copy()
np.fill_diagonal(sub, 0.0)

answers = {
    "n_kept": int(len(kept)),
    "kept_scores": scores[kept],
    "max_iou_among_kept": float(sub.max()),
    "n_kept_thresh_09": int(len(nms(boxes, scores, 0.9))),
}
```